(NeurIPS 2021) [M-FAC: Efficient Matrix-Free Approximations of Second-Order Information](https://arxiv.org/abs/2107.03356)

Информация о второй производной функции потерь — гессиане $H = \nabla^2_\theta \mathcal{L}$ — полезна сразу для нескольких задач:
- Прунинг по схеме Optimal Brain Surgeon: какие веса можно безболезненно обнулить и как пересчитать оставшиеся
- Преобуславливание (preconditioning) в задаче градиентного спуска: концепция natural gradient Амари, методы второго порядка
- Квантизация (например, GPTQ строится на тех же идеях)

Проблема в том, что для современных моделей гессиан физически нельзя хранить: для ResNet-50 это около 2.5 петабайт памяти. Поэтому работают с приближениями. Самая популярная — эмпирическая матрица Фишера:

$$
H \approx \hat{F} = \frac{1}{N} \sum_{i=1}^{N} \nabla \ell_i \, \nabla \ell_i^\top
$$

Чем хорошо такое приближение:
- можно считать "на лету" пока обучаешь модель backpropagation считаются все градиенты
- матрица положительно определена
- можно считать итеративно
- хорошая апроксимация, при модели к оптимуму становится неотличим от Гессиана $F \rightarrow H$

Это равенство использовалось еще в OBS. Данную замену предложил Amari в своих работах про оптимизацию через вычисление натурального градиента (учитывающего кривизну)<br>
[Amari, 1990, Natural Gradient](https://arxiv.org/pdf/2303.05473)

То есть сумма $N$ внешних произведений градиентов по отдельным примерам. Это всё ещё матрица $d \times d$, но с важной структурой: она представима как сумма рангов-1 плюс регуляризатор $\lambda I$ (демпфирование, нужное для обратимости).

### Что хочется уметь

В прунинге и оптимизации почти никогда не нужен сам гессиан или его обратный. Нужны:

1. Произведение обратного гессиана на вектор (IHVP, inverse-Hessian-vector product): $\hat{F}^{-1} v$.
2. Отдельные элементы обратного: $[\hat{F}^{-1}]_{ii}$ (диагональ для saliency в OBS) и $[\hat{F}^{-1}]_{ij}$.

Если научиться делать это без явного материализации матриц — выиграем колоссально по памяти и времени. Это и есть смысл слова *matrix-free* в названии.

---

## Глава 2. Краеугольный камень — формула Шермана–Моррисона

Когда матрица возмущена ранг-1 обновлением, её обратная пересчитывается аналитически:

$$
(A + u u^\top)^{-1} = A^{-1} - \frac{A^{-1} u u^\top A^{-1}}{1 + u^\top A^{-1} u}
$$

Если $\hat{F}_m = \lambda I + \frac{1}{m} \sum_{i=1}^{m} \nabla \ell_i \nabla \ell_i^\top$, можно построить обратную рекурсивно:

$$
\hat{F}_i^{-1} = \hat{F}_{i-1}^{-1} - \frac{\hat{F}_{i-1}^{-1} \nabla \ell_i (\hat{F}_{i-1}^{-1} \nabla \ell_i)^\top}{m + \nabla \ell_i^\top \hat{F}_{i-1}^{-1} \nabla \ell_i}, \qquad \hat{F}_0^{-1} = \lambda^{-1} I
$$

Прямая реализация хранит матрицу $\hat{F}^{-1}$ целиком — $O(d^2)$ памяти. Это ровно та точка, которую M-FAC улучшает: матрица никогда не строится явно.

### Что было до M-FAC

- WoodFisher (Singh & Alistarh, 2020) применяет эту рекурсию по блокам $B \times B$. Сложность $O(B d m)$ времени и $O(B d)$ памяти. На больших моделях $B$ всё равно мал и приближение грубое.
- K-FAC аппроксимирует Фишер кронекеровским произведением — работает хорошо, но не всегда соответствует реальности и неудобно для непростых слоёв.
- Диагональные приближения дешёвые, но качество хуже.

M-FAC обходит блочное ограничение: рекурсия Шермана–Моррисона выполняется *глобально* по всей размерности $d$, без блоков, и при этом без явных $d \times d$ матриц.

---

## Глава 3. Статический алгоритм (для прунинга)

Сцена: модель обучена, веса $\theta^*$ зафиксированы. Мы насчитали $m$ градиентов $\nabla \ell_1, \ldots, \nabla \ell_m$ (обычно $m$ от сотен до тысяч). Хотим уметь вычислять $\hat{F}_m^{-1} x$ для произвольного $x$ и читать отдельные элементы $[\hat{F}_m^{-1}]_{ij}$

### Ключевое наблюдение

Введём вспомогательные векторы:

$$
v_i := \hat{F}_{i-1}^{-1} \nabla \ell_i
$$

Разворачивая рекурсию Шермана–Моррисона, получаем удивительно простую формулу для IHVP:

$$
\hat{F}_m^{-1} x = \lambda^{-1} x - \sum_{j=1}^{m} v_j \cdot \frac{v_j^\top x}{m + \nabla \ell_j^\top v_j}
$$

Это и есть основная формула статического алгоритма. Заметьте: справа стоит линейная комбинация $x$ и $m$ векторов $v_j$. Никаких $d \times d$ матриц нет — только $d$-мерные векторы и скаляры.



### Алгоритм

**Предвычисление** (один раз):

1. Положить $v_1 = \lambda^{-1} \nabla \ell_1$.
2. Для $i = 2, \ldots, m$ вычислить $v_i = \hat{F}_{i-1}^{-1} \nabla \ell_i$, используя ту же формулу: $v_i = \lambda^{-1} \nabla \ell_i - \sum_{j=1}^{i-1} v_j \cdot \frac{v_j^\top \nabla \ell_i}{m + \nabla \ell_j^\top v_j}$.
3. Запомнить также скаляры $q_j = m + \nabla \ell_j^\top v_j$.

Шаг $i$ стоит $O(d \cdot i)$. Суммарно $O(d m^2)$.

**Вычисление IHVP** для нового $x$:

1. Посчитать $m$ скаляров $\alpha_j = v_j^\top x$.
2. Вернуть $\hat{F}_m^{-1} x = \lambda^{-1} x - \sum_j (\alpha_j / q_j) \, v_j$.

Стоимость $O(d m)$.

**Запрос одного элемента** $[\hat{F}_m^{-1}]_{ij}$:

$$
[\hat{F}_m^{-1}]_{ij} = e_i^\top \hat{F}_m^{-1} e_j = \lambda^{-1} \delta_{ij} - \sum_{k=1}^{m} \frac{(v_k)_i (v_k)_j}{q_k}
$$

где $(v_k)_i$ — $i$-я компонента вектора $v_k$. Это просто индексация — $O(m)$ на элемент. Для прунинга это ровно то, что нужно: можно посчитать diag$(\hat{F}^{-1})$ за $O(d m)$, не материализуя матрицу.

### Итог по сложности

| Операция | Время | Память |
|---|---|---|
| Предвычисление | $O(d m^2)$ | $O(d m)$ |
| IHVP $\hat{F}^{-1} x$ | $O(d m)$ | — |
| Один элемент $[\hat{F}^{-1}]_{ij}$ | $O(m)$ | — |

Сравните с прямой реализацией Вудбери ($\Omega(d^2 m)$) и блочным WoodFisher ($\Omega(d B m)$). При $m < B$ M-FAC получается в $B / m$ раз быстрее по тем же ресурсам — то есть на порядок и более.

---

## Глава 4. Применение к прунингу

Фреймворк OBS (Hassibi & Stork, 1993) говорит:

- Saliency (важность) веса $\theta_i$: $\rho(\theta_i) = \dfrac{\theta_i^2}{2 \, [\hat{F}^{-1}]_{ii}}$.
- Оптимальная коррекция оставшихся весов после удаления $\theta_i$: $\delta \theta = -\theta_i \, \hat{F}^{-1} e_i \, / \, [\hat{F}^{-1}]_{ii}$.

И там, и там — операции из таблицы выше. Авторы протестировали M-FAC на ResNet-50 и MobileNet-V1 на ImageNet:

- One-shot прунинг: M-FAC превосходит magnitude-based и слой-wise методы, причём качество монотонно растёт с $m$ до насыщения.
- Gradual прунинг: MobileNetV1 при 89% разреженности — M-FAC даёт 65.0% top-1 против 63.9% у WoodFisher и 62.9% у глобального magnitude.
- Скорость: один шаг прунинга MobileNet занимает у WoodFisher 60 минут, у M-FAC с теми же параметрами — 0.5 минуты (примерно в 100 раз быстрее).

Интересное побочное наблюдение из статьи: маска OBS обычно очень похожа на маску magnitude pruning. Основной выигрыш OBS даёт не за счёт *выбора* удаляемых весов, а за счёт *коррекции* оставшихся.

---

## Глава 5. Динамический алгоритм (для оптимизации)

В оптимизации ситуация другая: градиенты приходят онлайн. На шаге $t$ хочется иметь матрицу Фишера, построенную по последним $\Delta$ градиентам (скользящее окно). На каждом шаге нужно:

- Заменить старейший градиент на новый (slide window).
- Применить $\hat{F}^{-1}$ к текущему градиенту для предобусловленного шага: $\theta_{t+1} = \theta_t - \eta \hat{F}^{-1} \nabla \ell_t$.

Статический алгоритм здесь не подходит: $v_i$ зависят от *порядка* градиентов, и замена одного из них требует полного пересчёта.

### Идея динамического алгоритма

Записать IHVP в виде:

$$
\hat{F}_m^{-1} x = \lambda^{-1} x - \sum_{j=1}^{m} c_j^m \, \nabla \ell_j
$$

где коэффициенты $c_j^m$ — скаляры. Цель: устроить так, чтобы каждый $c_j^m$ выражался только через скалярные произведения $\nabla \ell_i^\top \nabla \ell_j$ и $\nabla \ell_i^\top x$. Тогда замена одного градиента в окне требует пересчёта только нескольких скалярных произведений плюс работа над $m \times m$ матрицами.

### Три рабочих матрицы

Алгоритм хранит:

- $G G^\top \in \mathbb{R}^{m \times m}$: $[GG^\top]_{ij} = \nabla \ell_i^\top \nabla \ell_j$, симметричная.
- $D \in \mathbb{R}^{m \times m}$: верхне-треугольная, $[D]_{ij} = \nabla \ell_i^\top \hat{F}_{i-1}^{-1} \nabla \ell_j$ для $i \le j$.
- $B \in \mathbb{R}^{m \times m}$: нижне-треугольная, коэффициенты разложения $\hat{F}_{i-1}^{-1} \nabla \ell_i = \sum_j [B]_{ij} \nabla \ell_j$.

Все три заполняются рекурсиями, выводимыми из формулы Шермана–Моррисона:

$$
[D]_{jk}^{(i)} = [D]_{jk}^{(i-1)} - \frac{[D]_{ji}^{(i-1)} \cdot [D]_{ik}^{(i-1)}}{m + [D]_{ii}^{(i-1)}}
$$

$$
[B]_{ii} = \lambda^{-1}, \qquad [B]_{ij} = -\sum_{k=j}^{i-1} \frac{[D]_{ki}}{m + [D]_{kk}} \, [B]_{kj}
$$

Заметьте: все рекурсии работают только с $m \times m$ матрицами и скалярами. Векторы $\nabla \ell_j \in \mathbb{R}^d$ хранятся, но не участвуют в этих внутренних пересчётах.

### Замена градиента в окне

Когда приходит новый градиент $\nabla \ell'$ и нужно вытеснить $\nabla \ell_k$:

1. Заменить строку $k$ матрицы $G$ — $O(d)$.
2. Пересчитать строку и столбец $k$ в $GG^\top$: умножение $G$ на новый градиент — $O(d m)$.
3. Пересчитать столбцы $j \ge k$ матрицы $D$ — $O(m^3)$ в худшем случае.
4. Пересчитать строки $i \ge k$ матрицы $B$ — $O(m^3)$.

Итого: $O(d m + m^3)$ на замену одного градиента.

### Вычисление IHVP

Дано $x$. Считаем $p = G x \in \mathbb{R}^m$ ($O(d m)$), потом запускаем рекурсию по $m \times m$:

$$
q_i = \frac{(\nabla \ell_i^\top \hat{F}_{i-1}^{-1} x)}{m + [D]_{ii}}, \qquad (\nabla \ell_j^\top \hat{F}_0^{-1} x) = \lambda^{-1} p_j
$$

$$
(\nabla \ell_j^\top \hat{F}_i^{-1} x) = (\nabla \ell_j^\top \hat{F}_{i-1}^{-1} x) - \frac{[D]_{ij}}{m + [D]_{ii}} \cdot (\nabla \ell_i^\top \hat{F}_{i-1}^{-1} x)
$$

Финал:

$$
\hat{F}_m^{-1} x = \lambda^{-1} x - \sum_{j=1}^{m} \left( \sum_{k=j}^{m} q_k [B]_{kj} \right) \nabla \ell_j
$$

Стоимость IHVP: $O(d m + m^2)$.

### Сводка сложности динамического алгоритма

| Операция | Время | Память |
|---|---|---|
| Замена одного градиента | $O(d m + m^3)$ | $O(d m + m^2)$ |
| IHVP | $O(d m + m^2)$ | — |

Сравните с GGT (Agarwal et al., 2019), где на каждом шаге делается SVD матрицы $m \times m$ — это $O(d m^2 + m^3)$. Авторы утверждают, что их подсчёт $D$ и $B$ работает в 10+ раз быстрее, чем SVD при $m = 1024$.

---

## Глава 6. Применение к оптимизации

M-FAC даёт предобусловленный SGD без момента:

$$
\theta_{t+1} = \theta_t - \eta_t \, \hat{F}_{[t-\Delta, t]}^{-1} \, \nabla \ell_t
$$

Гессиан оценивается по последним $\Delta$ градиентам (скользящее окно). Это, по сути, full-matrix natural gradient в духе Амари, но с дешёвой реализацией.

Экспериментально:

- На ResNet-20/CIFAR M-FAC даёт 92.34% top-1 против 91.78% у SGD и 92.17% у AdaHessian. Накладные расходы — около 55% над SGD.
- На WRN и MobileNet-V1 без подбора гиперпараметров M-FAC обыгрывает Adam и не отстаёт от SGD.
- На дообучении разреженных моделей оверхед падает до 5%, потому что эффективная размерность модели (число ненулевых весов) меньше.
- На малых BERT моделях (tiny, mini) на GLUE и SQuADv2 — M-FAC обычно лучше Adam при той же длине обучения.

Любопытно, что эффект особенно заметен в первых эпохах: алгоритм ведёт себя как настоящий метод второго порядка — быстро падает train loss.

---

## Глава 7. Где это сейчас живёт

M-FAC сам по себе — конкретная статья 2021 года, но идеи проросли в другие работы той же группы:

- SparseGPT (Frantar & Alistarh, 2023) — прунинг LLM. По сути, статический M-FAC, применённый послойно к задаче реконструкции активаций. Та же логика «обратный гессиан как сумма ранг-1 + матрично-свободная Вудбери», но в контексте «один слой за раз».
- GPTQ / OPTQ — квантизация LLM. Та же машинерия, только цель не обнулить веса, а округлить их.
- WoodFisher остаётся точкой отсчёта; M-FAC в каком-то смысле — его строго более быстрая версия.

## Что важно запомнить

- M-FAC — это не новая аппроксимация гессиана. Это новый эффективный *вычислительный* способ работать с уже известной аппроксимацией (эмпирический Фишер как сумма ранг-1).
- Численно M-FAC даёт *тот же* результат, что и прямое применение формулы Вудбери. Разница только в стоимости.
- Ключевая идея — никогда не материализовать $d \times d$ матрицу. Все рекурсии переписаны так, чтобы работать либо с $d$-мерными векторами и скалярами (статический случай), либо с $m \times m$ матрицами (динамический), где $m$ — размер окна или количество градиентов в выборке.
- Для прунинга получаем глобальный (не блочный) OBS-style апдейт по моделям с миллионами параметров. Для оптимизации — практически работоспособный second-order SGD.

## Дополнительное чтение

- Hassibi & Stork (1993). *Optimal Brain Surgeon* — фреймворк, для которого M-FAC даёт быструю реализацию.
- Singh & Alistarh (2020). *WoodFisher* — предшественник, блочное приближение.
- Agarwal et al. (2019). *GGT* — близкая идея для оптимизации, но через SVD.
- Frantar & Alistarh (2023). *SparseGPT* — масштабирование идей M-FAC до миллиардов параметров.
- Реализация: https://github.com/IST-DASLab/M-FAC